# Task 3: Customer Churn / Sales Trend Analysis
**Tools:** Python, Pandas, Matplotlib, Seaborn  
**Goal:** End-to-end analysis of e-commerce sales to find top customer segments, churn and revenue trends, then give actionable recommendations.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['figure.figsize'] = (10, 5)

## 2. Load the e-commerce dataset
This cell generates a realistic, reproducible e-commerce order dataset (15,000 orders, 1,500 customers, Jan 2025 to Sep 2026), so the notebook runs anywhere.

**To use a real dataset** (e.g. a Kaggle e-commerce CSV), replace this cell with `df = pd.read_csv('your_file.csv')` and rename columns to: `OrderID, CustomerID, OrderDate, Country, Category, UnitPrice, Quantity, Discount, Revenue`.

In [ ]:
rng = np.random.default_rng(42)
START, END_DAYS = pd.Timestamp('2025-01-01'), 638   # through 30 Sep 2026
n_customers, n_orders = 1500, 15000

customers = pd.DataFrame({
    'CustomerID': np.arange(1001, 1001 + n_customers),
    'SignupDay': rng.integers(0, 500, n_customers),
    'Country': rng.choice(['USA', 'UK', 'India', 'Germany', 'Canada', 'Australia'],
                          n_customers, p=[.35, .15, .20, .10, .10, .10]),
})
churner = rng.random(n_customers) < 0.30
customers['Lifetime'] = np.where(churner, rng.integers(30, 300, n_customers), 10_000)

weights = rng.gamma(shape=1.2, scale=1.0, size=n_customers)
weights /= weights.sum()
c = customers.iloc[rng.choice(n_customers, size=n_orders, p=weights)].reset_index(drop=True)

max_offset = np.minimum(c['Lifetime'], END_DAYS - c['SignupDay']).clip(lower=1)
day = c['SignupDay'] + rng.random(n_orders) * max_offset

cat_price = {'Electronics': (50, 600), 'Fashion': (15, 150), 'Home & Kitchen': (10, 200),
             'Beauty': (5, 80), 'Sports': (15, 250), 'Books': (5, 40)}
cats = rng.choice(list(cat_price), n_orders, p=[.20, .25, .20, .12, .13, .10])
price = np.array([rng.uniform(*cat_price[k]) for k in cats]).round(2)
qty = rng.integers(1, 5, n_orders)
disc = rng.choice([0, 0.05, 0.10, 0.20], n_orders, p=[.5, .2, .2, .1])

df = pd.DataFrame({
    'OrderID': np.arange(100000, 100000 + n_orders),
    'CustomerID': c['CustomerID'],
    'OrderDate': (START + pd.to_timedelta(day, unit='D')).dt.floor('D'),
    'Country': c['Country'],
    'Category': cats,
    'UnitPrice': price,
    'Quantity': qty,
    'Discount': disc,
})
df['Revenue'] = (df['UnitPrice'] * df['Quantity'] * (1 - df['Discount'])).round(2)
df = df.sort_values('OrderDate').reset_index(drop=True)
df.head()

In [ ]:
df.info()
df.describe()

## 3. Feature engineering and date extraction

In [ ]:
df['Year'] = df['OrderDate'].dt.year
df['Month'] = df['OrderDate'].dt.month
df['Quarter'] = df['OrderDate'].dt.quarter
df['DayOfWeek'] = df['OrderDate'].dt.day_name()
df['YearMonth'] = df['OrderDate'].dt.to_period('M')
df['YearQuarter'] = df['OrderDate'].dt.to_period('Q')
df['IsWeekend'] = df['OrderDate'].dt.dayofweek >= 5
df[['OrderDate', 'Year', 'Month', 'Quarter', 'DayOfWeek', 'YearMonth']].head()

## 4. Pivot tables and trend calculations

In [ ]:
monthly = df.groupby('YearMonth')['Revenue'].sum()
trend = monthly.to_frame('Revenue')
trend['MoM_Growth_%'] = (monthly.pct_change() * 100).round(2)
trend['3M_Moving_Avg'] = monthly.rolling(3).mean().round(2)
trend.tail(10)

In [ ]:
fig, ax = plt.subplots()
x = monthly.index.to_timestamp()
ax.plot(x, monthly.values, marker='o', color='#2E86AB', label='Monthly revenue')
ax.plot(x, trend['3M_Moving_Avg'].values, color='#E4572E', linewidth=2.5, label='3-month moving average')
ax.set_title('Monthly Revenue Trend')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
pivot_cat = df.pivot_table(values='Revenue', index='Category', columns='YearQuarter', aggfunc='sum')
fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(pivot_cat, annot=True, fmt='.0f', cmap='YlGnBu', ax=ax)
ax.set_title('Revenue by Category and Quarter')
plt.tight_layout()
plt.show()

In [ ]:
pivot_country = df.pivot_table(values='Revenue', index='Country', columns='Category', aggfunc='sum', margins=True)
pivot_country.round(0)

In [ ]:
order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow = df.groupby('DayOfWeek')['Revenue'].sum().reindex(order)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
dow.plot(kind='bar', color='#6A4C93', ax=axes[0])
axes[0].set_title('Revenue by Day of Week')
axes[0].set_xlabel('')
df.groupby('Category')['Revenue'].sum().sort_values().plot(kind='barh', color='#2E86AB', ax=axes[1])
axes[1].set_title('Revenue by Category')
plt.tight_layout()
plt.show()

## 5. Customer segmentation (RFM analysis)
RFM = **R**ecency (days since last order), **F**requency (number of orders), **M**onetary (total spend).

In [ ]:
snapshot = df['OrderDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg(
    Recency=('OrderDate', lambda s: (snapshot - s.max()).days),
    Frequency=('OrderID', 'nunique'),
    Monetary=('Revenue', 'sum'),
    Country=('Country', 'first'),
).reset_index()

rfm['R'] = pd.qcut(rfm['Recency'].rank(method='first'), 4, labels=[4, 3, 2, 1]).astype(int)
rfm['F'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
rfm['M'] = pd.qcut(rfm['Monetary'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)

rfm['Segment'] = np.select(
    [(rfm['R'] >= 3) & (rfm['F'] >= 3),
     (rfm['R'] >= 3),
     (rfm['R'] <= 2) & (rfm['F'] >= 3)],
    ['Champions', 'Active / New', 'At Risk (high value)'],
    default='Lost / Low value')
rfm.head()

In [ ]:
seg = rfm.groupby('Segment').agg(
    Customers=('CustomerID', 'count'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Orders=('Frequency', 'mean'),
    Total_Revenue=('Monetary', 'sum'),
).round(1)
seg['Revenue_Share_%'] = (seg['Total_Revenue'] / seg['Total_Revenue'].sum() * 100).round(1)
seg.sort_values('Total_Revenue', ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
seg['Customers'].plot(kind='pie', autopct='%1.0f%%', ax=axes[0], colormap='viridis')
axes[0].set_ylabel('')
axes[0].set_title('Customers by Segment')
seg['Total_Revenue'].sort_values().plot(kind='barh', color='#E4572E', ax=axes[1])
axes[1].set_title('Revenue by Segment')
plt.tight_layout()
plt.show()

## 6. Churn analysis
A customer is treated as **churned** if they have not ordered in the last 90 days.

In [ ]:
CHURN_DAYS = 90
rfm['Churned'] = rfm['Recency'] > CHURN_DAYS
print(f"Overall churn rate: {rfm['Churned'].mean():.1%}")

churn_country = rfm.groupby('Country')['Churned'].mean().sort_values(ascending=False) * 100
print()
print(churn_country.round(1))

In [ ]:
print(rfm.groupby('Churned')[['Frequency', 'Monetary']].mean().round(1))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(x=churn_country.index, y=churn_country.values, color='#E4572E', ax=axes[0])
axes[0].set_title('Churn Rate by Country (%)')
axes[0].set_xlabel('')
sns.boxplot(data=rfm, x='Churned', y='Monetary', hue='Churned', palette=['#2E86AB', '#E4572E'], legend=False, ax=axes[1])
axes[1].set_title('Customer Spend: Active vs Churned')
plt.tight_layout()
plt.show()

In [ ]:
top = rfm.sort_values('Monetary', ascending=False)
top20_share = top.head(int(len(top) * 0.2))['Monetary'].sum() / rfm['Monetary'].sum()
print(f"Top 20% of customers generate {top20_share:.1%} of revenue")
top.head(10)[['CustomerID', 'Country', 'Frequency', 'Monetary', 'Segment']]

## 7. Key business metrics

In [ ]:
best_cat = df.groupby('Category')['Revenue'].sum().idxmax()
best_country = df.groupby('Country')['Revenue'].sum().idxmax()
best_day = dow.idxmax()
best_month = monthly.idxmax()
at_risk = seg.loc['At Risk (high value)', 'Customers'] if 'At Risk (high value)' in seg.index else 0
at_risk_rev = seg.loc['At Risk (high value)', 'Total_Revenue'] if 'At Risk (high value)' in seg.index else 0

print(f"Total revenue          : {df['Revenue'].sum():,.0f}")
print(f"Total orders           : {df['OrderID'].nunique():,}")
print(f"Average order value    : {df['Revenue'].mean():,.2f}")
print(f"Top category           : {best_cat}")
print(f"Top country            : {best_country}")
print(f"Best day of week       : {best_day}")
print(f"Best month             : {best_month}")
print(f"Churn rate (90 days)   : {rfm['Churned'].mean():.1%}")
print(f"At-risk high-value customers: {at_risk} (revenue at stake: {at_risk_rev:,.0f})")

## 8. Actionable business recommendations
*(Use the numbers printed above when presenting these.)*

1. **Win back at-risk high-value customers.** They have bought often but have gone quiet. Send a personalised discount or reminder email before they are fully lost.
2. **Reward Champions.** Launch a loyalty or VIP programme (early access, free shipping) since this small group drives a large share of revenue.
3. **Invest in the top category and country.** Focus ad budget and inventory on the best performers, and test promotions in weaker countries with high churn.
4. **Time campaigns to the best days and months.** Schedule promotions and email sends on peak days to maximise conversion.
5. **Fix retention.** Churned customers order less often, so add a post-purchase journey (review request, replenishment reminders, a second-order coupon) to push new buyers to repeat.
6. **Watch month-over-month growth.** If the moving average is flat or falling, acquisition is not replacing churned customers, so balance spend between acquisition and retention.

## Conclusion
This analysis used date features, pivot tables, trend calculations and RFM segmentation to find who the best customers are, who is churning, and where revenue comes from, ending in clear recommendations.